# FinIA-Flex — Paso 5: Uniendo RAG + LLM

Caso Práctico Unidad 1, materia Generative IA, IEP.

Hasta ahora tenía dos piezas sueltas: el índice del Paso 3 (busca la política) y el prompt
del Paso 4 (redacta el reporte, pero yo le pegaba el contexto a mano). Aquí las conecto para
que la búsqueda sea automática — le doy solo los datos financieros y el sistema hace el resto
solo.

Esto es literalmente lo que es RAG: recuperación + generación en un mismo flujo.


## 1. Instalación

Mismas dependencias de los Pasos 3 y 4, más la API key de Groq.

In [ ]:
!pip install -q langchain langchain-community langchain-chroma langchain-text-splitters chromadb sentence-transformers groq

from getpass import getpass
import os

os.environ["GROQ_API_KEY"] = getpass("Ingresar API key de Groq: ")

## 2. Cargar el índice vectorial ya construido

En lugar de reconstruir el índice desde cero, se carga la base vectorial que se guardó en
Google Drive al final del Paso 3. Si esa carpeta no existe todavía, es necesario correr
primero el notebook del Paso 3 completo.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma

CHROMA_PATH = "/content/drive/MyDrive/FinIA-Flex/finia_flex_chroma_db"

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectorstore = Chroma(
    persist_directory=CHROMA_PATH,
    embedding_function=embeddings,
)

print(f"Índice cargado con {vectorstore._collection.count()} fragmentos disponibles.")

## 3. Prompt maestro

El mismo prompt validado y corregido en el Paso 4 (incluye la regla de grounding que evita
inventar responsables).


In [ ]:
SYSTEM_PROMPT = """Eres un analista financiero senior de FlexParts Manufacturing MX,
especializado en control de costos de manufactura. Tu tarea es generar reportes ejecutivos
de variación presupuestal para Gerencia, a partir de datos de presupuesto vs. gasto real y
del contexto de políticas internas que se te proporcione.

RAZONAMIENTO INTERNO (no lo muestres en la respuesta final, solo úsalo para pensar):
1. Los indicadores de clasificación, umbral y variación sostenida ya vienen calculados por
   el sistema en el bloque "INDICADORES CALCULADOS POR EL SISTEMA". NO los recalcules, NO los
   contradigas, NO compares tú mismo montos contra umbrales — usa esos valores tal cual.
2. Con base en esos indicadores ya calculados, verifica si el contexto de políticas
   proporcionado incluye una regla que corresponda a lo que los indicadores señalan (umbral
   excedido, variación sostenida, o ninguno). Si el contexto no incluye una política aplicable
   a lo que indican los indicadores, dilo explícitamente — nunca inventes un umbral o una
   regla que no esté en el contexto.
3. Distingue causas internas (atendibles por el responsable del centro de costo) de causas
   externas (fuera de su control), cuando el contexto lo permita.
4. Solo después de este análisis, redacta el reporte final.

ESTRUCTURA OBLIGATORIA DE LA RESPUESTA FINAL:
1. Resumen Ejecutivo (máximo 3 líneas)
2. Diagnóstico por Centro de Costo (variación en MXN y %, clasificación, causa probable)
3. Alertas de Política (solo si el contexto proporcionado activa alguna; si no, escribir
   "Sin alertas de política en el contexto disponible")
4. Recomendación (una acción concreta y accionable por cada hallazgo relevante)
5. Responsable y Siguiente Paso

REGLAS DE GROUNDING:
- Usa exclusivamente los datos numéricos y el contexto de políticas que se te proporcionen.
- Si citas una política o un umbral, debe provenir textualmente del contexto recibido.
- Si el contexto no cubre algo que sería útil mencionar, indica la limitación en vez de
  completar con supuestos.
- NUNCA inventes nombres de personas, cargos o responsables. El campo "Responsable" debe
  llenarse únicamente con el valor recibido en los DATOS de entrada, copiado tal cual. Si el
  campo Responsable no viene incluido en los DATOS, escribe exactamente "No especificado en
  los datos proporcionados" — no propongas un nombre, cargo o departamento por tu cuenta bajo
  ninguna circunstancia.
- NUNCA recalcules ni contradigas los valores del bloque "INDICADORES CALCULADOS POR EL
  SISTEMA". Si ese bloque indica que el umbral NO se excedió, no afirmes lo contrario aunque
  el monto te parezca alto; si indica que sí se excedió, no lo minimices.

TONO: profesional, directo, sin tecnicismos innecesarios. El reporte debe ser legible para
un Gerente de Planta que no es especialista financiero. Evita juicios de valor sobre las
personas; evalúa procesos y resultados.
"""

EJEMPLO_FEW_SHOT_ENTRADA = """
DATOS:
Centro de costo: Línea de Producción 2
Categoría: Mantenimiento
Presupuesto: $70,000 MXN | Real: $87,200 MXN (Septiembre)
Histórico: Julio +18.6%, Agosto +20.0%, Septiembre +24.6%
Responsable: Coordinador de Mantenimiento - J. Salinas

CONTEXTO DE POLÍTICAS RECUPERADO:
"Cuando una categoría de gasto en un mismo centro de costo presenta una variación positiva
(sobrecosto) durante 3 meses consecutivos o más, el responsable debe presentar un plan
correctivo formal a Gerencia, independientemente de si cada mes individual superó o no el
umbral de aprobación." (Política POL-FIN-001, sección 4)
"""

EJEMPLO_FEW_SHOT_SALIDA = """
1. Resumen Ejecutivo
Mantenimiento en Línea de Producción 2 muestra sobrecosto sostenido por tercer mes
consecutivo, activando la regla de variación sostenida de la Política POL-FIN-001.

2. Diagnóstico por Centro de Costo
- Línea de Producción 2 / Mantenimiento: variación de +$17,200 MXN (+24.6%) en septiembre.
  Clasificación: significativa. Tendencia sostenida desde julio (+18.6%, +20.0%, +24.6%).

3. Alertas de Política
Se activa la regla de variación sostenida (POL-FIN-001, sección 4): 3 meses consecutivos de
sobrecosto en la misma categoría y centro de costo requieren plan correctivo formal a
Gerencia, independientemente del monto individual de cada mes.

4. Recomendación
Solicitar al Coordinador de Mantenimiento un plan correctivo formal antes del cierre del
siguiente mes, desagregando el gasto entre mantenimiento correctivo y preventivo para
identificar si el sobrecosto responde a fallas puntuales o a un patrón estructural.

5. Responsable y Siguiente Paso
Responsable: Coordinador de Mantenimiento - J. Salinas.
Siguiente paso: presentar plan correctivo formal a Gerencia — fecha límite sugerida: cierre
del mes en curso.
"""

print("Prompt maestro y ejemplo few-shot cargados.")

## 4. Búsqueda automática

Esta función toma el centro de costo y la categoría, arma una pregunta, y busca sola en el
índice — ya no tengo que copiar y pegar el contexto a mano.

Aquí me encontré un problema real: en mi primera versión, la consulta solo decía algo como
"variación de 70.0%", sin más contexto. Con eso, en 2 de mis 3 escenarios de prueba el sistema
no encontraba el fragmento correcto (por ejemplo, no hallaba la regla del umbral de $50,000
para Mantenimiento, aunque sí estaba en el índice — lo verifiqué en el Paso 3). El problema
era que la consulta no traía las palabras clave necesarias. Lo arreglé metiendo el histórico
completo en la consulta y subiendo de 2 a 3 los fragmentos que trae (`k`).


In [ ]:
def recuperar_contexto(centro_costo: str, categoria: str, resumen_situacion: str,
                        historico: str = "", k: int = 3) -> str:
    """
    Busca en el índice vectorial los fragmentos de política más relevantes
    para un caso financiero dado, y los devuelve como texto de contexto.

    Se incluye el histórico completo en la consulta (no solo la variación del mes
    actual), ya que reglas como la de "variación sostenida" o los umbrales de
    aprobación por categoría solo se recuperan de forma confiable si la consulta
    contiene las palabras clave asociadas (p. ej. "consecutivos", "aprobación").
    """
    consulta = (
        f"Política de aprobación y control de gasto para la categoría {categoria} "
        f"en el centro de costo {centro_costo}. Situación actual: {resumen_situacion}. "
        f"Histórico reciente: {historico}. "
        f"¿Qué umbral de aprobación o regla de variación sostenida aplica?"
    )
    resultados = vectorstore.similarity_search(consulta, k=k)

    fragmentos_texto = []
    for r in resultados:
        fuente = r.metadata["source"].split("/")[-1]
        fragmentos_texto.append(f'({fuente})\n"{r.page_content}"')

    return "\n\n".join(fragmentos_texto)


# Prueba rápida de la función de recuperación, de forma aislada
contexto_prueba = recuperar_contexto(
    centro_costo="Mantenimiento",
    categoria="Mantenimiento",
    resumen_situacion="gasto puntual muy por encima del presupuesto en un solo mes",
    historico="Pico aislado en julio, no observado en meses anteriores",
)
print(contexto_prueba)

## 4.5. Calculando los indicadores yo mismo (en código)

Este es el hallazgo más importante del proyecto: en una prueba con los 3 escenarios, el
modelo comparó mal las cifras — usó el gasto real total ($87,200) en vez de la variación
sobre presupuesto ($17,200) para ver si excedía el umbral de $50,000 de Mantenimiento.
Resultado: citó la regla equivocada.

Me di cuenta de que no tiene caso pedirle a un LLM que haga esta aritmética — no es
confiable para eso. Así que aquí calculo yo mismo, en Python, todo lo que antes le pedía
inferir al modelo (umbral excedido, variación sostenida, clasificación), y se lo entrego ya
resuelto. El modelo ya no decide nada numérico, solo redacta.


In [ ]:
UMBRALES_APROBACION_MXN = {
    "Materia Prima": 80000,
    "Mano de Obra Directa": 40000,
    "Energía": 30000,
    "Mantenimiento": 50000,
    "Logística/Fletes": 35000,
}


def clasificar_variacion(variacion_pct: float) -> str:
    """Clasifica la variación según los rangos definidos en POL-FIN-002, sección 3."""
    if variacion_pct >= 10:
        return "Variación significativa"
    elif variacion_pct >= 3:
        return "Variación moderada"
    elif variacion_pct >= -2.9:
        return "Dentro de rango normal"
    elif variacion_pct >= -10:
        return "Ahorro saludable"
    else:
        return "Ahorro atípico"


def calcular_racha_sobrecosto(variacion_pct_actual: float, historico_mensual: list) -> int:
    """
    Cuenta cuántos meses consecutivos de sobrecosto (variación positiva) hay,
    incluyendo el mes actual, recorriendo el histórico del más reciente al más antiguo.
    historico_mensual: lista de tuplas (mes, variacion_pct), en orden cronológico.
    """
    racha = 1 if variacion_pct_actual > 0 else 0
    if racha == 0:
        return 0
    for _, pct in reversed(historico_mensual):
        if pct > 0:
            racha += 1
        else:
            break
    return racha


def calcular_indicadores(categoria: str, variacion_mxn: float, variacion_pct: float,
                          historico_mensual: list = None) -> dict:
    """
    Calcula, en código (no en el LLM), los indicadores que antes el modelo debía
    inferir por sí mismo: clasificación de la variación, si excede el umbral de
    aprobación de su categoría, y si hay variación sostenida 3+ meses consecutivos.
    """
    historico_mensual = historico_mensual or []

    umbral = UMBRALES_APROBACION_MXN.get(categoria)
    excede_umbral = (umbral is not None) and (variacion_mxn > umbral)

    racha = calcular_racha_sobrecosto(variacion_pct, historico_mensual)
    variacion_sostenida = racha >= 3

    return {
        "clasificacion": clasificar_variacion(variacion_pct),
        "umbral_categoria_mxn": umbral,
        "excede_umbral": excede_umbral,
        "racha_meses_sobrecosto": racha,
        "variacion_sostenida": variacion_sostenida,
    }


# Prueba rápida con el caso del hallazgo (Escenario 1)
indicadores_prueba = calcular_indicadores(
    categoria="Mantenimiento",
    variacion_mxn=17200,
    variacion_pct=24.6,
    historico_mensual=[("Julio", 18.6), ("Agosto", 20.0)],
)
print(indicadores_prueba)

## 5. La función completa

Esta es la función principal: recibe los datos de un caso, calcula los indicadores (Sección
4.5), busca el contexto (Sección 4), arma todo el prompt, y devuelve el reporte. De aquí en
adelante, usar el copiloto es simplemente llamar a esta función.


In [ ]:
from groq import Groq

client = Groq()

def generar_reporte(centro_costo: str, categoria: str, presupuesto: float, real: float,
                     historico: str, historico_mensual: list = None,
                     responsable: str = None) -> str:
    """
    Flujo completo RAG + LLM: calcula indicadores de forma determinística,
    recupera contexto de políticas automáticamente, y genera el reporte ejecutivo.
    """
    variacion_mxn = real - presupuesto
    variacion_pct = variacion_mxn / presupuesto * 100

    indicadores = calcular_indicadores(categoria, variacion_mxn, variacion_pct, historico_mensual)

    resumen_situacion = f"variación de {variacion_pct:.1f}% ({variacion_mxn:,.0f} MXN)"
    contexto = recuperar_contexto(centro_costo, categoria, resumen_situacion, historico=historico)

    responsable_texto = responsable if responsable else "No especificado en los datos proporcionados"

    umbral_texto = (
        f"${indicadores['umbral_categoria_mxn']:,.0f} MXN" if indicadores["umbral_categoria_mxn"]
        else "sin umbral definido para esta categoría"
    )

    caso_entrada = f"""
DATOS:
Centro de costo: {centro_costo}
Categoría: {categoria}
Presupuesto: ${presupuesto:,.0f} MXN | Real: ${real:,.0f} MXN
Variación: {variacion_pct:.1f}% (${variacion_mxn:,.0f} MXN)
Histórico: {historico}
Responsable: {responsable_texto}

INDICADORES CALCULADOS POR EL SISTEMA (usa estos valores tal cual, no los recalcules ni los
contradigas — fueron calculados por código, no por ti):
- Clasificación de la variación: {indicadores['clasificacion']}
- Umbral de aprobación de la categoría "{categoria}": {umbral_texto}
- ¿La variación de este mes excede el umbral de su categoría?: {"SÍ" if indicadores['excede_umbral'] else "NO"}
- Meses consecutivos de sobrecosto (incluyendo el actual): {indicadores['racha_meses_sobrecosto']}
- ¿Aplica la regla de variación sostenida (3+ meses consecutivos de sobrecosto)?: {"SÍ" if indicadores['variacion_sostenida'] else "NO"}

CONTEXTO DE POLÍTICAS RECUPERADO (automático):
{contexto}
"""

    respuesta = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": EJEMPLO_FEW_SHOT_ENTRADA},
            {"role": "assistant", "content": EJEMPLO_FEW_SHOT_SALIDA},
            {"role": "user", "content": caso_entrada},
        ],
        temperature=0.3,
    )

    return respuesta.choices[0].message.content


print("Función generar_reporte() lista para usarse (con indicadores precalculados).")

## 6. Probando los 3 escenarios completos

Corro los tres casos de mi dataset (Paso 1), ahora sin pasar nada de contexto a mano.


In [ ]:
# Escenario 1: sobrecosto sostenido — Línea de Producción 2 / Mantenimiento
reporte_1 = generar_reporte(
    centro_costo="Línea de Producción 2",
    categoria="Mantenimiento",
    presupuesto=70000,
    real=87200,
    historico="Julio +18.6%, Agosto +20.0%, Septiembre +24.6%",
    historico_mensual=[("Julio", 18.6), ("Agosto", 20.0)],
    responsable="Gerente de Línea 2 - M. Torres",
)
print("=" * 90)
print("ESCENARIO 1: Sobrecosto sostenido")
print("=" * 90)
print(reporte_1)

In [ ]:
# Escenario 2: ahorro consistente — Línea de Producción 1 / Materia Prima
reporte_2 = generar_reporte(
    centro_costo="Línea de Producción 1",
    categoria="Materia Prima",
    presupuesto=620000,
    real=565100,
    historico="Variación entre -7% y -9% durante todo el año",
    historico_mensual=[],
    responsable="Gerente de Línea 1 - R. Hernández",
)
print("=" * 90)
print("ESCENARIO 2: Ahorro consistente")
print("=" * 90)
print(reporte_2)

In [ ]:
# Escenario 3: violación de política puntual — Mantenimiento / Mantenimiento, julio
reporte_3 = generar_reporte(
    centro_costo="Mantenimiento",
    categoria="Mantenimiento",
    presupuesto=85000,
    real=144500,
    historico="Pico aislado en julio, no observado en meses anteriores",
    historico_mensual=[],
    responsable="Coordinador de Mantenimiento - J. Salinas",
)
print("=" * 90)
print("ESCENARIO 3: Violación de política puntual")
print("=" * 90)
print(reporte_3)

## 7. Verificación cruzada

Para cada uno de los 3 reportes generados, confirmar:

- **Escenario 1:** ¿el reporte cita la regla de variación sostenida a 3 meses (POL-FIN-001)?
- **Escenario 2:** ¿el reporte reconoce que -8.8% NO activa la regla de ahorro atípico
  (que aplica por debajo de -10%), sin confundirla?
- **Escenario 3:** ¿el reporte detecta que la variación en MXN supera el umbral de $50,000 en
  Mantenimiento y cita explícitamente esa regla?

Si los tres puntos se cumplen, el flujo RAG + LLM está funcionando correctamente de extremo a
extremo, y la recuperación automática está trayendo el fragmento correcto en cada caso — no
solo el fragmento más genérico o el primero del índice.


---
## Resumen técnico (Paso 5)

**Qué hice:** uní el índice del Paso 3 con el prompt del Paso 4 en dos funciones —
`recuperar_contexto()` y `generar_reporte()`. Ya no tengo que buscar ni pegar contexto a mano.

Decisiones:
- La consulta se arma con centro de costo, categoría, variación e histórico — no solo el
  nombre de la categoría — para que traiga fragmentos más relevantes.
- Recupero los 3 fragmentos más cercanos (`k=3`).
- Todo el cálculo (umbral, clasificación, racha) va en Python, no en el LLM.

**Los dos hallazgos de este paso:**
1. La búsqueda no traía el fragmento correcto en 2 de 3 casos porque la consulta no tenía
   suficientes palabras clave — lo arreglé metiendo el histórico completo y subiendo k.
2. El modelo comparó mal el gasto real contra el umbral (en vez de la variación) y citó la
   regla equivocada — lo arreglé moviendo ese cálculo a Python (Sección 4.5).

Los 3 reportes de la Sección 6 y la verificación de la Sección 7 son la evidencia de que ya
funciona bien de principio a fin.

Siguiente: Paso 6, el fine-tuning.
